# Гибридный поиск

Notebook загружает полную мету, FAISS и BM25S-шарды, объединяет кандидатов через RRF и выполняет CrossEncoder-реранкинг.

# Импорты и настройки

In [ ]:
%pip install -r requirements.txt

In [1]:
import os
import re
import time
import pickle
import gc
import bm25s
import faiss
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder

In [2]:
BM25_CHUNK = 1000000
path_to_load = "cache_le_finale2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)
print("Загрузка BGE-M3...")

path_to_model = os.getenv("EMBEDDING_MODEL_NAME", "BAAI/bge-m3")
path_to_reranker = os.getenv("RERANKER_MODEL_NAME", "BAAI/bge-reranker-v2-m3")
EMBED_MODEL = SentenceTransformer(path_to_model, device=DEVICE)
EMBED_MODEL.to(DEVICE)
DIM = EMBED_MODEL.get_sentence_embedding_dimension()
print(f"Готово. Размерность вектора: {DIM}")
reranker = CrossEncoder(path_to_reranker, device=DEVICE)

cpu
Загрузка BGE-M3...


Готово. Размерность вектора: 1024


# Функции, необходимые для запуска

Токенизация и создание эмбеддингов

In [3]:
def tokenize(text):
    """Разбивает текст на токены."""
    return re.findall(r"[а-яёa-z0-9]+", text.lower())


def embed(texts, batch_size=48, log_every=50000):
    """Создаёт эмбеддинги для списка текстов."""
    all_embeddings = []
    total = len(texts)
    start_total = time.time()
    last_log_time = start_total
    processed = 0
    last_logged_processed = 0
    next_log = log_every
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        emb = EMBED_MODEL.encode(
            batch,
            normalize_embeddings=True,
            convert_to_numpy=True,
            batch_size=batch_size,
            device=DEVICE,
            chunk_size=200,
            show_progress_bar=False
        )
        all_embeddings.append(emb)
        processed += len(batch)
        if processed >= next_log or processed == total:
            now = time.time()
            last_log_time = now
            last_logged_processed = processed
            while next_log <= processed:
                next_log += log_every

        # Очистка каждые ~100к эмбеддингов
        if len(all_embeddings) >= 10:
            all_embeddings = [np.vstack(all_embeddings)]
    return np.vstack(all_embeddings).astype("float32")

Загрузка данных и индексов

In [4]:
def load_meta(path=path_to_load):
    """Загружает полную мету из файла."""
    global documents, doc_ids, tokenized_corpus, id_to_index, req_descs, msg_pprb_chats, req_reg_dates
    with open(f"{path}/meta.pkl", "rb") as f:
        meta = pickle.load(f)
    documents = meta["documents"]
    doc_ids = meta["doc_ids"]
    tokenized_corpus = meta["tokenized_corpus"]
    id_to_index = meta["id_to_index"]
    req_descs = meta["req_descs"]
    msg_pprb_chats = meta["msg_pprb_chats"]
    req_reg_dates = meta.get("req_reg_dates", [None] * len(documents))
    print("мета успешно загружена!")
    print(f"Количество документов: {len(documents)}")


def load_indices(path=path_to_load, to_gpu=False):
    """Загружает индекс FAISS."""
    faiss_index = faiss.read_index(f"{path}/faiss_index")
    if to_gpu:
        try:
            res = faiss.StandardGpuResources(); faiss_index = faiss.index_cpu_to_gpu(res, 0, faiss_index)
        except Exception as e: print("GPU error:", e)
    print("FAISS index загружен")
    return faiss_index


def load_bm25s_shards(bm25_dir):
    """Загружает BM25S-шарды."""
    bm25_indexes = []  # сюда будем складывать загруженные индексы
    # находим все шарды в папке
    shards = sorted([f for f in os.listdir(bm25_dir) if f.startswith("shard_")])
    for shard in shards:  # проходим по всем шардам
        shard_id = int(shard.split("_")[1])  # из имени "shard_8" например, достаем номер шарда
        offset = shard_id * BM25_CHUNK  # считаем сдвиг шарда: для шарда с номером 1 он равен 1000 000 при размере шарда = 1000к; для шарда с номером 2 (третьим) он равен 2000 000
        bm25_index = bm25s.BM25.load(f"{bm25_dir}/{shard}", load_corpus=False)  # загружаем индекс в память
        bm25_indexes.append((bm25_index, offset))  # сохраняем пару (индекс, сдвиг)
    print("BM25 shards загружены:", len(bm25_indexes))
    return bm25_indexes

Преобразование диапазона дат в маску, где True - документ подходит, False - документ не подходит

In [5]:
def build_date_mask(date_range=None):
    """Строит маску документов по диапазону дат."""
    if date_range is None:
        return None
    start_date = str(date_range[0])
    end_date = str(date_range[1])
    return np.array([date is not None and start_date <= str(date) <= end_date for date in req_reg_dates], dtype=bool)

Получение кандидатов через FAISS + BM25S + RRF

In [6]:
K_RRF = 60
ALPHA = 0.3  # вес BM25 в финальном скоре
TOP_K_RERANK = 50
DEFAULT_TOP_K = 250
SCORE_THRESHOLD = 0.5


def retrieve_hybrid_adaptive(query, faiss_idx, bm25_indexes, target_k, date_range=None):
    """Получает кандидатов через FAISS, BM25S и RRF."""
    # ищем в ближайших соседей
    faiss_k = 2048  # больше 2048 нельзя - фаисс при добавлении возвращать максимум 2048 ближайших
    bm25_total_k = int(faiss_k * 0.67)
    num_shards = len(bm25_indexes)
    bm25_k_per_shard = int(np.ceil(bm25_total_k / max(1, num_shards)))  # привязан к числу шардов (если огромное кол-во шардов, то bm25 не затмевает фаисс)
    date_mask = build_date_mask(date_range)

    # Поиск по faiss
    faiss_ranks = {}  # словарь типа {doc_id: rank} - ранги фаисс
    q_emb = embed([query]).astype("float32")
    if date_range is None:
        # поиск по faiss; I - индексы найденных документов, их ответ выдается от большего скора к меньшему, то ранг это документа 0 (двойка); D - скор
        D, I = faiss_idx.search(q_emb, min(faiss_k, faiss_idx.ntotal))
    else:
        # позиция, где date_mask=True (документ подходит по дате)
        allowed_ids = np.ascontiguousarray(np.flatnonzero(date_mask).astype("int64"))
        if len(allowed_ids) == 0:
            return []
        # Создаем встроенный селектор, он разрешает искать ответы только среди allowed_ids
        selector = faiss.IDSelectorBatch(allowed_ids)
        params = faiss.SearchParametersIVF()  # Создаем параметры поиска для IVF-индекса
        params.nprobe = faiss_idx.nprobe  # Сохраняем текущее значение nprobe индекса; nprobe определяет, в скольких IVF кластерах будет выполняться поиск
        params.sel = selector  # Подключаем фильтр разрешенных ID
        D, I = faiss_idx.search(q_emb, min(faiss_k, len(allowed_ids)), params=params)
    for rank, idx in enumerate(I[0], 1):  # enumerate(I[0], 1) возьмет индексы (например, idx=15, idx=333, idx=72) и присвоит им ранги, начиная с 1 (rank=1, rank=2, rank=3)
        idx = int(idx)
        if idx < 0 or idx >= len(doc_ids):  # защита от битых индексов
            continue
        cid = doc_ids[idx]  # cid - id кандидата; подхватываем по индексу эмбеддинга (позиция в массиве эмбеддингов), найденного фаиссом, индекс самого документа (настоящий id обращения)
        faiss_ranks[cid] = rank  # записываем пару "кандидат от фаисс: его ранг"

    # Поиск по BM-25
    bm25_ranks = {}  # словарь типа {doc_id: rank} - ранги bm25
    query_tokens = tokenize(query)
    # проходимся по заранее загруженным BM25 шардам; bm25_index - индекс шарда, offset - позиция первого документа шарда в общем наборе данных
    for bm25_index, offset in bm25_indexes:
        shard_size = int(bm25_index.scores["num_docs"])
        if date_range is None:
            results, scores = bm25_index.retrieve([query_tokens], k=min(bm25_k_per_shard, shard_size))
            local_mask = None
        else:
            # Вырезаем из общей маски часть, соответствующую текущему BM-25 шарду
            local_mask = date_mask[offset:offset + shard_size].astype("float32")
            allowed_count = int(local_mask.sum())
            if allowed_count == 0:  # Если в этом шарде нет документов подходящей даты, полностью пропускаем его
                continue
            results, scores = bm25_index.retrieve([query_tokens], k=min(bm25_k_per_shard, allowed_count), weight_mask=local_mask, show_progress=False)
        # Проходимся по локальным id внутри шарда - ранги
        for rank, local_id in enumerate(results[0], 1):
            local_id = int(local_id)
            if local_id < 0 or local_id >= shard_size:
                continue
            # Защита от нулевых результатов под маской
            if local_mask is not None and local_mask[local_id] == 0:
                continue
            global_idx = offset + local_id  # переводим локальный индекс шарда в глобальный индекс документа (например в шарде 1 у меня документы с id 1000 000
            # до 1999 999); retriever же вернул локальные индексы; в shard 1 локальный индекс = 15, 66, 20001, хотя глобальный id 15 соответственно
            # документу из первого шарда, поэтому нужен offset: global_idx = offset + local_id; для local_ids = 15 имеем globalidx = 15 + 1000 000 = 1000 015
            # Пропускаем невалидные индексы (проверяем, что глобальная позиция существует)
            if global_idx < 0 or global_idx >= len(doc_ids):
                continue
            cid = doc_ids[global_idx]  # Получаем настоящий id обращения
            # Если документ еще не встречался, сохраняем его ранг (добавляем пару "кандидат от bm25 внутри текущего шарда: его ранг")
            # Если документ встречался несколько раз (в шарде несколько раз один документ попался) - оставляем его самый высокий BM25 rank
            if cid not in bm25_ranks or rank < bm25_ranks[cid]:
                bm25_ranks[cid] = rank

    # RRF fusion: объединяем кандидатов faiss и bm25
    all_ids = set(faiss_ranks) | set(bm25_ranks)
    # делаем слияние фаисса и bm25: слияние рангов, а не скоров; RRF = сумма(1/(k+r(d))); где r(d) - ранг (позиция) документа d в списке c (начиная с 1); k - константа (обычно 60)
    # у нас 2 слагаемых: одно по рангам faiss, другое по рангам bm25; также можем установить через ALPHA важность bm25 и фаисс
    # Если документа нет в BM25, берем ранг 999 (очень плохой ранг) - можно и 1501 взять (тк в фаисс 1500 кандидатов отбирается, чтобы не было искусственного отрыва), но это уже не важно
    # Важно - 999 или 1500 - нужному к этому моменту уже замужа
    fused = {cid: ALPHA * (1 / (K_RRF + bm25_ranks.get(cid, 999))) +
                  (1 - ALPHA) * (1 / (K_RRF + faiss_ranks.get(cid, 999)))
             for cid in all_ids}
    # полученная оценка будет от 0 до 0.02327 (при объединении двух поисковых систем) с учетом K=60
    sorted_cands = [cid for cid, _ in sorted(fused.items(), key=lambda x: x[1], reverse=True)]  # сортируем документы по полученному "общему" скору; id кандидатов в списке
    return sorted_cands[:target_k]

Реранкер

In [7]:
def rerank(query: str, candidates):
    """Переранжирует кандидатов с помощью CrossEncoder."""
    rerank_inputs = [(query, documents[id_to_index[cid]]) for cid in candidates]  # формируем пары (запрос, документ)
    raw_scores = reranker.predict(rerank_inputs, batch_size=32)  # реранкер дает скоры, соответствия каждой паре (запрос-документ)
    raw_scores = np.array(raw_scores)
    scores = 1 / (1 + np.exp(-raw_scores))

    results = []
    for cid, score in zip(candidates, scores):
        idx = id_to_index[cid]
        results.append({
            "id": cid,
            "Короткое описание": req_descs[idx],
            "Транскрибация диалога": msg_pprb_chats[idx],
            "score": float(score),
            "date": req_reg_dates[idx],
        })
    return sorted(results, key=lambda x: x["score"], reverse=True)

Функция поиска финальная

In [8]:
def search_pipeline(query, faiss_idx, bm25_indexes, top_k=None, date_range=None):
    """Выполняет полный гибридный поиск."""
    target_k = top_k if top_k else DEFAULT_TOP_K  # если пользователь не указал, сколько запросов он хочет получить - получим DEFAULT_TOP_K
    # Берем больше кандидатов, чем требуется вернуть (при большом top_k кандидатов также становится больше)
    candidates_k = max(target_k*2, TOP_K_RERANK)
    candidates = retrieve_hybrid_adaptive(query=query, faiss_idx=faiss_idx, bm25_indexes=bm25_indexes, target_k=candidates_k, date_range=date_range)
    if not candidates:
        return pd.DataFrame()
    # Реранжируем всех полученных кандидатов уже ограниченным значением candidates_k
    reranked = rerank(query, candidates)
    # Если пользователь явно указал, сколько ответов выводить, возвращаем top_k лучших без фильтрации по порогу
    if top_k is not None:
        return pd.DataFrame(reranked[:top_k])[["id", "Короткое описание", "Транскрибация диалога", "score", "date"]]
    else:
        # если пользователь НЕ указал, сколько ответов вывести, выводим все релевантные (выше порога по скору)
        # reranked содержит DEFAULT_TOP_K значений максимум (если все прошли порог по скору)
        filtered = [result for result in reranked if result["score"] >= SCORE_THRESHOLD]
        return pd.DataFrame(filtered)[["id", "Короткое описание", "Транскрибация диалога", "score", "date"]]

# Запуск

In [9]:
load_meta()

мета успешно загружена!
Количество документов: 100


In [10]:
bm25_indexes = load_bm25s_shards(f"{path_to_load}/bm25s_shards2")

BM25 shards загружены: 1


In [11]:
faiss_loaded = load_indices(to_gpu=False)

FAISS index загружен


In [12]:
MIN_DATE = "0000-01-01"
MAX_DATE = "9999-12-31"
# Демонстрация: пять релевантных обращений по оплате картой только за 2026 год.
res = search_pipeline(
    query="не проходит оплата банковской картой",
    faiss_idx=faiss_loaded,
    bm25_indexes=bm25_indexes,
    top_k=5,
    date_range=("2026-01-01", "2026-12-31")
    # date_range=(MIN_DATE, "2026-01-20") # до определенной даты
    # date_range=("2026-01-20", MAX_DATE) # после определенной даты
    # date_range=("2026-01-20", "2026-01-20") # в течение одного дня
)
res

,id,Короткое описание,Транскрибация диалога,score,date
0,REQ-0001,Не проходит оплата картой в магазине,Покупатель несколько раз приложил банковскую к...,0.706355,2026-01-08
1,REQ-0006,Не проходит оплата за границей,Банковская карта отклоняется в зарубежном мага...,0.702713,2026-03-19
2,REQ-0010,Не проходит оплата после перевыпуска карты,"Новая карта активирована, однако терминалы маг...",0.640443,2026-05-28
3,REQ-0014,Не проходит оплата виртуальной картой,"Реквизиты виртуальной карты введены верно, но ...",0.618303,2026-08-05
4,REQ-0008,Не удаётся оплатить подписку картой,Сервис не может списать ежемесячный платёж с б...,0.555736,2026-04-21


In [13]:
res.to_csv('results.csv', sep='~', index=False, encoding='utf-8')
print('Выполнено успешно!')

Выполнено успешно!
